# 🌊 Resonance — Physics AI
**Gemma 4 Good Hackathon**

Pipeline:
1. Verifica GPU
2. Instala dependências
3. Autentica Kaggle e baixa Gemma 4 E2B
4. Carrega modelo (inferência)
5. Coleta dataset do arXiv com respostas reais
6. Fine-tuning com Unsloth + QLoRA
7. Benchmarks
8. Demo pública via Gradio

> **Pré-requisito:** `Runtime > Change runtime type > T4 GPU`

## 0. Verifica GPU

In [ ]:
import torch
assert torch.cuda.is_available(), '❌ GPU não detectada. Vá em Runtime > Change runtime type > T4 GPU'
gpu = torch.cuda.get_device_properties(0)
print(f'✅ GPU: {gpu.name}')
print(f'   VRAM total:      {gpu.total_memory / 1e9:.1f} GB')
print(f'   VRAM disponível: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

## 1. Instala dependências

In [ ]:
!pip install -q kagglehub
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q arxiv sympy scipy numpy matplotlib astropy gradio
print('✅ Dependências instaladas')

## 2. Autentica Kaggle e baixa o Gemma 4 E2B

In [ ]:
# 1. Acesse kaggle.com/settings/api
# 2. Role até 'Legacy API Credentials' > 'Create Legacy API Key'
# 3. Faça upload do kaggle.json quando o seletor abrir

import os
from google.colab import files

print('📂 Faça upload do seu kaggle.json')
uploaded = files.upload()

os.makedirs('/root/.config/kaggle', exist_ok=True)
with open('/root/.config/kaggle/kaggle.json', 'wb') as f:
    f.write(list(uploaded.values())[0])
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('✅ Kaggle autenticado')

import kagglehub
print('\n⏳ Baixando Gemma 4 E2B...')
MODEL_PATH = kagglehub.model_download('google/gemma-4/transformers/gemma-4-e2b')
print(f'✅ Modelo em: {MODEL_PATH}')

## 3. Carrega o modelo base (inferência)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type       = 'nf4',
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config = quant_config,
    device_map          = 'cuda',
)
base_model.eval()
print('✅ Modelo carregado')
print(f'   VRAM usada: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## 4. Função de inferência

In [ ]:
SYSTEM_PROMPT = (
    'Você é Resonance, assistente científico especializado em física. '
    'Responda sempre em dois níveis:\n'
    '**Nível 1 — Intuição:** explicação acessível sem formalismo excessivo.\n'
    '**Nível 2 — Formal:** tratamento matemático completo com equações.'
)

def generate(question, max_new_tokens=500, system=SYSTEM_PROMPT):
    """Gera resposta usando o formato nativo de chat do Gemma."""
    prompt = (
        f'<bos><start_of_turn>user\n'
        f'[SYSTEM]: {system}\n\n'
        f'{question}<end_of_turn>\n'
        f'<start_of_turn>model\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = base_model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = 0.3,
            do_sample          = True,
            eos_token_id       = tokenizer.eos_token_id,
            pad_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.3,
        )
    input_len = inputs['input_ids'].shape[1]
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)

# Teste rápido
print(generate('O que é difração de ondas?', max_new_tokens=200))

## 5. Coleta dataset com respostas reais
Gera respostas usando o modelo base — evita os placeholders que causam overfitting.

In [ ]:
import json, time, random
from pathlib import Path
import arxiv
from tqdm.notebook import tqdm

Path('data').mkdir(exist_ok=True)

CATEGORIES = ['physics.optics','astro-ph.GA','astro-ph.SR','physics.class-ph','physics.em-ph']
TEMPLATES = {
    'optic': ['Como funciona {c} em óptica? Derive e visualize.',
              'Explique {c} e mostre a equação que o descreve.'],
    'astro': ['O que é {c} em astronomia e como é modelado matematicamente?'],
    'class': ['Derive a equação de movimento para {c}.',
              'Explique {c} intuitivamente e formalmente.'],
    'em-ph': ['Como as equações de Maxwell descrevem {c}?'],
}

def fetch_papers(max_per=60):
    client = arxiv.Client(page_size=50, delay_seconds=3.0, num_retries=3)
    papers, seen = [], set()
    for cat in CATEGORIES:
        s = arxiv.Search(query=f'cat:{cat}', max_results=max_per,
                         sort_by=arxiv.SortCriterion.Relevance)
        for r in client.results(s):
            if r.entry_id in seen: continue
            ab = r.summary.strip().replace('\n',' ').split()
            if len(ab) < 50: continue
            papers.append({'id': r.entry_id.split('/')[-1], 'title': r.title.strip(),
                           'abstract': ' '.join(ab[:400]), 'categories': r.categories})
            seen.add(r.entry_id)
        time.sleep(1)
    print(f'✅ {len(papers)} papers coletados')
    return papers

def make_question(p):
    cats = ' '.join(p['categories']).lower()
    words = [w for w in p['title'].split() if len(w) > 5]
    c = ' '.join(words[:3]) if words else p['title'][:50]
    tmpl = ['Explique {c} e mostre a matemática por trás.']
    for key, tl in TEMPLATES.items():
        if key in cats: tmpl = tl; break
    return random.choice(tmpl).format(c=c)

def make_sample(p):
    q    = make_question(p)
    resp = generate(q, max_new_tokens=400)
    if len(resp.split()) < 30: return None
    return {
        'messages': [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': q},
            {'role': 'assistant', 'content': resp},
        ],
        'metadata': {'arxiv_id': p['id'], 'title': p['title']}
    }

papers  = fetch_papers(max_per=60)
samples = []
# Limita a 150 papers — suficiente e não esgota a sessão do Colab
for p in tqdm(papers[:150], desc='Gerando dataset'):
    s = make_sample(p)
    if s: samples.append(s)

random.shuffle(samples)
split = int(len(samples) * 0.9)
for name, data in [('train', samples[:split]), ('eval', samples[split:])]:
    with open(f'data/{name}.jsonl', 'w', encoding='utf-8') as f:
        for s in data: f.write(json.dumps(s, ensure_ascii=False) + '\n')

print(f'✅ Dataset: {split} train | {len(samples)-split} eval')

## 6. Fine-tuning com Unsloth + QLoRA

In [ ]:
# Libera memória do modelo de inferência antes de carregar com Unsloth
import gc
del base_model
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM disponível: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

In [ ]:
import json
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MAX_SEQ_LEN  = 2048
OUTPUT_DIR   = 'checkpoints'
HUB_MODEL_ID = 'seu-usuario/resonance-gemma4-physics'  # ← mude para o seu username HF

# Carrega com Unsloth para fine-tuning
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_PATH,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = torch.bfloat16,
    load_in_4bit   = True,
)
ft_model = FastLanguageModel.get_peft_model(
    ft_model,
    r              = 16,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha     = 16,
    lora_dropout   = 0.1,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
ft_tokenizer = get_chat_template(ft_tokenizer, chat_template='gemma')
print(f'✅ Modelo carregado | Parâmetros treináveis: {ft_model.num_parameters(only_trainable=True):,}')

In [ ]:
# Formata dataset
def format_sample(s):
    parts = []
    for m in s.get('messages', []):
        role, content = m.get('role',''), m.get('content') or ''
        if role == 'system':
            parts.append(f'<start_of_turn>user\n[SYSTEM]: {content}\n')
        elif role == 'user':
            parts.append(f'<start_of_turn>user\n{content}<end_of_turn>\n')
        elif role == 'assistant':
            parts.append(f'<start_of_turn>model\n{content}<end_of_turn>\n')
    return '<bos>' + ''.join(parts) + '<eos>'

def load_jsonl(path):
    data = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip(): data.append(json.loads(line))
    return data

train_ds = Dataset.from_list([{'text': format_sample(s)} for s in load_jsonl('data/train.jsonl')])
eval_ds  = Dataset.from_list([{'text': format_sample(s)} for s in load_jsonl('data/eval.jsonl')])
train_ds = train_ds.filter(lambda x: len(x['text']) > 200)
eval_ds  = eval_ds.filter(lambda x:  len(x['text']) > 200)
print(f'✅ Dataset: {len(train_ds)} train | {len(eval_ds)} eval')

In [ ]:
# Treina
# Correções aplicadas vs versão anterior:
#   - TrainingArguments → SFTConfig
#   - tokenizer → processing_class
#   - evaluation_strategy → eval_strategy
#   - warmup_ratio → warmup_steps
#   - dataset_text_field e max_seq_length agora ficam no SFTConfig

trainer = SFTTrainer(
    model            = ft_model,
    processing_class = ft_tokenizer,
    train_dataset    = train_ds,
    eval_dataset     = eval_ds,
    dataset_num_proc = 2,
    packing          = False,
    args = SFTConfig(
        output_dir                  = OUTPUT_DIR,
        max_seq_length              = MAX_SEQ_LEN,
        dataset_text_field          = 'text',
        num_train_epochs            = 3,
        per_device_train_batch_size = 1,
        per_device_eval_batch_size  = 1,
        gradient_accumulation_steps = 8,
        learning_rate               = 5e-5,
        warmup_steps                = 50,
        lr_scheduler_type           = 'cosine',
        weight_decay                = 0.01,
        max_grad_norm               = 0.3,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        save_steps                  = 50,
        eval_steps                  = 50,
        eval_strategy               = 'steps',
        save_strategy               = 'steps',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_loss',
        optim                       = 'adamw_8bit',
        report_to                   = 'none',
        seed                        = 42,
    ),
)

stats = trainer.train()
print(f'✅ Fine-tuning concluído!')
print(f'   Loss final:  {stats.training_loss:.4f}')
print(f'   Tempo total: {stats.metrics["train_runtime"]:.0f}s')

In [ ]:
# Salva pesos
ft_model.save_pretrained('model/lora')
ft_tokenizer.save_pretrained('model/lora')
print('✅ Pesos LoRA salvos em model/lora')

# Publica no HuggingFace (necessário para o prize do Unsloth)
from huggingface_hub import login
hf_token = input('Cole seu token HuggingFace (ou Enter para pular): ').strip()
if hf_token:
    login(token=hf_token)
    ft_model.push_to_hub_gguf(
        HUB_MODEL_ID, ft_tokenizer,
        quantization_method = ['q4_k_m', 'f16'],
        token               = hf_token,
    )
    print(f'✅ Publicado em: https://huggingface.co/{HUB_MODEL_ID}')
else:
    print('⚠️  Publicação pulada — pesos salvos localmente em model/lora')

## 7. Benchmarks (obrigatório para o prize do Unsloth)

In [ ]:
# Recarrega modelo base para benchmarks e demo
import gc
del ft_model
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config = quant_config,
    device_map          = 'cuda',
)
base_model.eval()

QUESTIONS = [
    'Explique difração e derive a condição de mínimos para fenda simples.',
    'Qual é a equação de onda eletromagnética no vácuo?',
    'Como calcular o redshift gravitacional de uma estrela de nêutrons?',
    'Descreva a lei de Snell a partir do princípio de Fermat.',
    'O que as equações de Maxwell representam fisicamente?',
]

results = []
for i, q in enumerate(QUESTIONS):
    resp = generate(q, max_new_tokens=350)
    results.append({'id': i+1, 'question': q, 'response': resp})
    print(f'Q{i+1}: {q[:60]}')
    print(f'R:  {resp[:300]}\n')

with open('benchmarks.json', 'w', encoding='utf-8') as f:
    json.dump({'model': 'gemma-4-e2b', 'results': results}, f, ensure_ascii=False, indent=2)
print('✅ Benchmarks salvos em benchmarks.json')

## 8. Demo pública com Gradio
Gera link público `gradio.live` para os juízes acessarem.

In [ ]:
import io, base64, tempfile
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.fft import fft, fftfreq
import gradio as gr

# ── Ferramentas científicas ───────────────────────────────────────────────

def run_sympy(expression, operation, variable='x'):
    try:
        x, t, k, omega = sp.symbols('x t k omega', real=True)
        var_map = {'x':x,'t':t,'k':k,'omega':omega}
        var  = var_map.get(variable, sp.Symbol(variable, real=True))
        expr = sp.sympify(expression, locals={
            'x':x,'t':t,'k':k,'omega':omega,
            'cos':sp.cos,'sin':sp.sin,'exp':sp.exp,'sqrt':sp.sqrt,'pi':sp.pi})
        ops = {'simplify': sp.simplify, 'diff': lambda e: sp.diff(e, var),
               'integrate': lambda e: sp.integrate(e, var), 'expand': sp.expand}
        res = ops.get(operation, sp.simplify)(expr)
        return f'Resultado: {res}\nLaTeX: ${sp.latex(res)}$'
    except Exception as e:
        return f'Erro SymPy: {e}'

def run_scipy_and_plot(sim_type, params):
    t_eval = np.linspace(0, 10, 1000)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), facecolor='#0d1117')
    BLUE, RED = '#58a6ff', '#f78166'

    def style(ax, title='', xl='', yl=''):
        ax.set_facecolor('#0d1117')
        ax.tick_params(colors='#8b949e')
        for s in ax.spines.values(): s.set_edgecolor('#30363d')
        ax.grid(True, color='#21262d', lw=0.5, alpha=0.8)
        if title: ax.set_title(title, color='#e6edf3', pad=8)
        if xl:    ax.set_xlabel(xl, color='#8b949e')
        if yl:    ax.set_ylabel(yl, color='#8b949e')

    if sim_type == 'oscillator':
        w   = float(params.get('omega', 2*np.pi))
        sol = solve_ivp(lambda t,y:[y[1],-w**2*y[0]], (0,10), [1,0], t_eval=t_eval)
        freqs = fftfreq(1000, d=10/1000)
        spec  = np.abs(fft(sol.y[0]))
        axes[0].plot(sol.t, sol.y[0], color=BLUE, lw=1.8)
        style(axes[0], 'Oscilador Harmônico', 't (s)', 'x(t)')
        axes[1].fill_between(freqs[:500], spec[:500], alpha=0.3, color=RED)
        axes[1].plot(freqs[:500], spec[:500], color=RED, lw=1.5)
        style(axes[1], 'Espectro (FFT)', 'Hz', 'Amplitude')

    elif sim_type == 'wave_packet':
        x_arr = np.linspace(-10, 10, 1000)
        k0, sigma = float(params.get('k0',5)), float(params.get('sigma',1))
        psi = np.exp(-x_arr**2/(2*sigma**2)) * np.cos(k0*x_arr)
        axes[0].fill_between(x_arr, psi, alpha=0.2, color=BLUE)
        axes[0].plot(x_arr, psi, color=BLUE, lw=2)
        style(axes[0], 'Pacote de Onda', 'x', 'ψ(x)')
        spec = np.abs(fft(psi))
        freqs = fftfreq(1000)
        axes[1].plot(freqs[:500], spec[:500], color=RED, lw=1.5)
        style(axes[1], 'Espectro do Pacote', 'k', 'Amplitude')

    elif sim_type == 'orbital':
        e = float(params.get('eccentricity', 0.5))
        a = float(params.get('semi_major', 1.0))
        th = np.linspace(0, 2*np.pi, 1000)
        axes[0].plot(a*np.cos(th), a*np.sqrt(1-e**2)*np.sin(th), color=BLUE, lw=2)
        axes[0].scatter([0],[0], color=RED, s=120, zorder=5, label='Estrela')
        axes[0].set_aspect('equal')
        axes[0].legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9')
        style(axes[0], f'Órbita Kepleriana (e={e})', 'x (UA)', 'y (UA)')
        # Segunda figura: velocidade orbital
        r = a*(1-e**2)/(1+e*np.cos(th))
        axes[1].plot(th*180/np.pi, r, color=RED, lw=2)
        style(axes[1], 'Distância ao foco', 'θ (graus)', 'r (UA)')

    else:
        x_arr = np.linspace(-2*np.pi, 2*np.pi, 500)
        axes[0].plot(x_arr, np.cos(x_arr)*np.exp(-0.1*x_arr**2), color=BLUE, lw=2)
        style(axes[0], 'Visualização', 'x', 'f(x)')
        axes[1].axis('off')

    plt.tight_layout(pad=1.2)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
    plt.close(fig)
    buf.seek(0)
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.png')
    tmp.write(buf.read())
    tmp.close()
    return tmp.name


# ── Interface Gradio ──────────────────────────────────────────────────────

CSS = """
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@300;400&family=Syne:wght@700;800&display=swap');
:root{--bg:#0d1117;--bg2:#161b22;--border:#30363d;--text:#c9d1d9;--dim:#8b949e;--blue:#58a6ff;--red:#f78166;--green:#56d364;}
body,.gradio-container{background:var(--bg)!important;font-family:'JetBrains Mono',monospace!important;color:var(--text)!important;}
.title h1{font-family:'Syne',sans-serif!important;font-size:2.4rem;font-weight:800;
  background:linear-gradient(135deg,#58a6ff,#56d364);-webkit-background-clip:text;-webkit-text-fill-color:transparent;
  text-align:center;margin:0;}
.title p{color:var(--dim);text-align:center;font-size:.8rem;letter-spacing:.06em;margin-top:.3rem;}
#chatbot{background:var(--bg2)!important;border:1px solid var(--border)!important;min-height:460px;}
#send{background:var(--blue)!important;color:#0d1117!important;font-family:'Syne',sans-serif!important;font-weight:700!important;border:none!important;}
#plot{background:var(--bg2)!important;border:1px solid var(--border)!important;}
"""

EXAMPLES = [
    'Explique difração de ondas e derive a condição de mínimos.',
    'Como as equações de Maxwell descrevem a propagação da luz?',
    'O que é um pacote de onda quântico e como ele se comporta?',
    'Derive a lei de Snell a partir do princípio de Fermat.',
    'Como calcular o redshift gravitacional de uma estrela de nêutrons?',
    'Explique o efeito Doppler e calcule o desvio para v=0.1c.',
]

# Mapa de palavras-chave para simulações automáticas
SIM_MAP = {
    'oscilador':  ('oscillator', {}),
    'harmônico':  ('oscillator', {}),
    'oscil':      ('oscillator', {}),
    'pacote':     ('wave_packet', {'k0': 5, 'sigma': 1}),
    'wave packet':('wave_packet', {'k0': 5, 'sigma': 1}),
    'órbita':     ('orbital', {'eccentricity': 0.5}),
    'orbital':    ('orbital', {'eccentricity': 0.5}),
    'kepleriana': ('orbital', {'eccentricity': 0.5}),
}

def respond(msg, history):
    if not msg.strip():
        return history, history, None, ''

    # Gera resposta do modelo
    resp = generate(msg, max_new_tokens=500)
    history = history + [[msg, resp]]

    # Detecta se deve gerar visualização
    msg_lower = msg.lower()
    plot_path = None
    for keyword, (sim_type, params) in SIM_MAP.items():
        if keyword in msg_lower:
            plot_path = run_scipy_and_plot(sim_type, params)
            break

    return history, history, plot_path, ''


with gr.Blocks(css=CSS, title='Resonance — Physics AI') as demo:
    gr.HTML('<div class="title"><h1>Resonance</h1><p>GEMMA 4 E2B · PHYSICS REASONING · SYMPY · SCIPY · MATPLOTLIB</p></div>')

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(elem_id='chatbot', height=480,
                                 show_copy_button=True, render_markdown=True, label='')
            with gr.Row():
                user_in   = gr.Textbox(placeholder='Pergunte sobre física — ondas, óptica, eletromagnetismo, astronomia...',
                                       show_label=False, lines=2, scale=5)
                with gr.Column(scale=1, min_width=80):
                    send_btn  = gr.Button('Enviar', elem_id='send', variant='primary')
                    clear_btn = gr.Button('Limpar')
        with gr.Column(scale=2):
            plot_out = gr.Image(elem_id='plot', label='Visualização Científica', height=400)

    gr.Examples(EXAMPLES, inputs=user_in, label='Experimente')
    history_st = gr.State([])

    send_btn.click(respond,  [user_in, history_st], [chatbot, history_st, plot_out, user_in])
    user_in.submit(respond,  [user_in, history_st], [chatbot, history_st, plot_out, user_in])
    clear_btn.click(lambda: ([],[],None), outputs=[chatbot, history_st, plot_out])

# Link público para os juízes
demo.launch(share=True, show_error=True)

---
## ✅ Checklist de submissão
- [ ] Pesos publicados no HuggingFace Hub
- [ ] `benchmarks.json` gerado e no repositório
- [ ] Link da demo Gradio funcionando
- [ ] Repositório público no GitHub
- [ ] Vídeo de 3 minutos no YouTube
- [ ] Kaggle Writeup preenchido e submetido